# Notebook 4: Room-Temperature PET-Degrading Candidates

Filters PlasticDB to entries where:
- `plastic == 'PET'`
- `thermophilic == False` (mesophilic / room-temperature conditions)

Scores each organism by evidence quality and ranks them as candidates
for room-temperature PETase discovery.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "plastic-biodegradation-analysis"))
from src.data_loader import load_plasticdb
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = load_plasticdb()

pet_df = df[df["plastic"] == "PET"].copy()
meso   = pet_df[pet_df["thermophilic"] == False].copy()

print(f"Total PlasticDB entries:      {len(df):,}")
print(f"PET entries:                  {len(pet_df):,}")
print(f"Mesophilic PET entries:       {len(meso):,}")
print(f"With sequence:                {meso['has_sequence'].sum():,}")
print(f"With named enzyme:            {meso['has_enzyme'].sum():,}")
print(f"With GenBank ID:              {meso['has_genbank'].sum():,}")


## Evidence scoring

In [ ]:
meso["score"] = (
    meso["has_sequence"].astype(int) * 3 +
    meso["has_genbank"].astype(int)  * 2 +
    meso["has_enzyme"].astype(int)   * 2 +
    (meso["year"] >= 2018).astype(int)
)

org_scores = (
    meso.groupby("organism")
    .agg(
        n_entries       = ("organism", "count"),
        max_score       = ("score", "max"),
        has_sequence    = ("has_sequence", "any"),
        has_enzyme      = ("has_enzyme", "any"),
        isolation_envs  = ("isolation_environment", lambda x: "; ".join(sorted(x.dropna().unique()))),
        isolation_locs  = ("isolation_location", lambda x: "; ".join(sorted(x.dropna().unique()))),
        first_year      = ("year", "min"),
        last_year       = ("year", "max"),
    )
    .reset_index()
    .sort_values("max_score", ascending=False)
)

print(f"Unique mesophilic PET organisms: {len(org_scores):,}")
print("\nTop 20 by evidence score:")
print(org_scores.head(20)[["organism","max_score","n_entries","has_sequence","has_enzyme"]].to_string(index=False))


## Isolation environments of the candidate pool

In [ ]:
env_counts = meso["isolation_environment"].value_counts().head(12)
print(env_counts.to_string())


## Visualisation

In [ ]:
top20 = org_scores.head(20).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Room-Temperature PET Candidates (mesophilic, PET, thermophilic=No)", fontsize=13, fontweight="bold")

ax = axes[0]
colors_seq = ["#4CAF50" if s else "#9E9E9E" for s in top20["has_sequence"]]
ax.barh(top20["organism"][::-1], top20["max_score"][::-1],
        color=colors_seq[::-1], edgecolor="black", linewidth=0.5)
ax.set_xlabel("Evidence score")
ax.set_title("Top 20 Mesophilic PET-Degrading Organisms", fontweight="bold")
patches = [mpatches.Patch(color="#4CAF50", label="Has sequence"),
           mpatches.Patch(color="#9E9E9E", label="No sequence")]
ax.legend(handles=patches, fontsize=9)
ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
vals = [len(pet_df), len(meso), int(meso["has_sequence"].sum()), int(meso["has_enzyme"].sum())]
labels = ["All PET entries","Mesophilic PET","Mesophilic + sequence","Mesophilic + enzyme"]
colors = ["#9E9E9E","#2196F3","#4CAF50","#FF9800"]
bars = ax.bar(labels, vals, color=colors, edgecolor="black")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(v),
            ha="center", fontsize=12, fontweight="bold")
ax.set_ylabel("Entries"); ax.set_title("Research Funnel", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/04_room_temp_candidates.png", dpi=150, bbox_inches="tight")
plt.show()

# Save report
org_scores.to_csv("../outputs/reports/04_room_temp_pet_candidates.csv", index=False)
print("Saved figure and report.")
